# ATLAS ML Progress Report

This notebook summarizes the current status of the code in this project. The current workflow is an end-to-end baseline for reconstructing four particle-level angular observables from reco-level event information.

**Current stage:** dataset preparation and baseline angular-regression model.

**Not yet implemented:** final signal/background classification model.

## 1. Project Goal

The long-term physics goal is to use reconstructed collision-event information to help distinguish signal-like events from background-like events.

At the current stage, we are first training a regression model:

```text
reco-level event features -> particle-level angular variables
```

The four angular variables are expected to encode useful event geometry, spin-correlation, and decay-topology information. These variables can later be used as physics-informed inputs to a signal/background classifier.

## 2. Current Repository Structure

The current project contains two main scripts:

- `src/prepare_dataset.py`: reads the ROOT file, selects events, matches reco-level and particle-level information, and writes a compressed NumPy dataset.
- `src/train_mlp.py`: trains a baseline multi-layer perceptron regression model and saves the trained model checkpoint.

Generated artifacts:

- `data/ttbarh_angles.npz`: selected dataset used for training and validation.
- `outputs/mlp_angles.pt`: saved PyTorch model checkpoint.

In [ ]:
from pathlib import Path

project_dir = Path.cwd()
for path in [
    project_dir / "src" / "prepare_dataset.py",
    project_dir / "src" / "train_mlp.py",
    project_dir / "data" / "ttbarh_angles.npz",
    project_dir / "outputs" / "mlp_angles.pt",
]:
    print(f"{path}: {'exists' if path.exists() else 'missing'}")

## 3. Dataset Preparation

The dataset preparation script reads the input ROOT file:

```text
/Users/michaelquu/Desktop/input_4Angle_10x.root
```

It reads two trees:

- `reco`: reconstructed-level variables.
- `particleLevel`: particle-level angular targets.

The script matches events using the event number, applies event selection, removes non-finite rows, and saves the final arrays into `data/ttbarh_angles.npz`.

## 4. Input Features

The current model uses 22 reco-level input features per selected event.

These include pileup information, channel flags, lepton kinematics, b-jet kinematics, spectator-jet kinematics, neutrino information, and MET.

In [ ]:
import numpy as np

data = np.load("data/ttbarh_angles.npz", allow_pickle=True)

X = data["X"]
y = data["y"]
weight = data["weight"]
feature_names = data["feature_names"]
target_names = data["target_names"]

print("X shape:", X.shape)
print("y shape:", y.shape)
print("weights shape:", weight.shape)
print("\nInput features:")
for i, name in enumerate(feature_names):
    print(f"{i:02d}: {name}")

The 22 input features are:

```text
actual_mu, average_mu,
is_ejets, is_mujets,
lep_e, lep_pt, lep_eta, lep_phi,
bjet_e, bjet_pt, bjet_eta, bjet_phi,
specjet_e, specjet_pt, specjet_eta, specjet_phi,
nu_e, nu_pt, nu_eta, nu_phi,
met_met, met_phi
```

## 5. Regression Targets

The current model predicts four particle-level angular observables:

```text
PL_cos_theta_lep_NOSYS
PL_cos_theta_star_lep_NOSYS
PL_phi_lep_NOSYS
PL_phi_star_lep_NOSYS
```

These are not signal/background labels. They are intermediate physics observables that may later help a classifier distinguish signal from background.

In [ ]:
print("Regression targets:")
for i, name in enumerate(target_names):
    values = y[:, i]
    print(
        f"{i}: {name} | "
        f"mean={values.mean():.4f}, std={values.std():.4f}, "
        f"min={values.min():.4f}, max={values.max():.4f}"
    )

## 6. Event Selection and Weights

The dataset preparation applies the following selections:

- `pass_SUBcommon_NOSYS`
- `passNuReco_NOSYS`
- electron+jets or muon+jets channel
- all input features are finite
- all target variables are finite
- event weight is finite

The event weight is formed from:

```text
weight_mc_NOSYS
* weight_pileup_NOSYS
* weight_leptonSF_tight_NOSYS
* weight_jvt_effSF_NOSYS
* weight_ftag_effSF_GN2v01_Continuous_NOSYS
```

The training script clips the weights at the 99.5th percentile and normalizes them by the mean before training.

In [ ]:
print("Selected events:", len(X))
print("Number of input features:", X.shape[1])
print("Number of regression targets:", y.shape[1])
print("\nRaw weight summary:")
print("mean:", weight.mean())
print("std:", weight.std())
print("min:", weight.min())
print("max:", weight.max())
print("99.5 percentile:", np.percentile(weight, 99.5))

## 7. Baseline Model Architecture

The current model is a simple fully connected multi-layer perceptron:

```text
22 inputs -> 128 hidden units -> 128 hidden units -> 4 outputs
```

The hidden-layer width of 128 is a baseline hyperparameter. It was not derived from a physics formula. It is a standard first-choice model size that gives the network enough capacity to learn nonlinear combinations of the 22 reco-level inputs without making the model excessively large for the current dataset.

The model uses ReLU activations and is trained with a weighted mean-squared-error loss.

In [ ]:
import torch
from torch import nn

class MLP(nn.Module):
    def __init__(self, n_features: int, n_targets: int) -> None:
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_features, 128),
            nn.ReLU(),
            nn.Linear(128, 128),
            nn.ReLU(),
            nn.Linear(128, n_targets),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)

model = MLP(X.shape[1], y.shape[1])
print(model)

## 8. Training Setup

The training script uses:

- 80/20 train-validation split
- random seed: 7
- batch size: 512
- optimizer: AdamW
- learning rate: `1e-3`
- weight decay: `1e-4`
- default epochs: 30

The input features are standardized using the training-set mean and standard deviation. The saved checkpoint contains the model weights, target names, feature names, and normalization constants.

## 9. Current Validation Performance

Using the saved model checkpoint and the same train-validation split, the current validation performance is approximately:

| Target | MAE | RMSE | Bias | Correlation |
|---|---:|---:|---:|---:|
| `PL_cos_theta_lep_NOSYS` | 0.255 | 0.329 | -0.014 | 0.760 |
| `PL_cos_theta_star_lep_NOSYS` | 0.243 | 0.314 | -0.016 | 0.648 |
| `PL_phi_lep_NOSYS` | 0.985 | 1.323 | -0.013 | 0.590 |
| `PL_phi_star_lep_NOSYS` | 0.907 | 1.232 | 0.073 | 0.606 |

The strongest current performance is on `PL_cos_theta_lep_NOSYS`. The phi variables and `theta_star` variables need additional treatment and diagnostic work.

In [ ]:
def weighted_mse(pred: torch.Tensor, target: torch.Tensor, weight: torch.Tensor) -> torch.Tensor:
    per_event = torch.mean((pred - target) ** 2, dim=1)
    return torch.sum(per_event * weight) / torch.sum(weight)

rng = np.random.default_rng(7)
indices = rng.permutation(len(X))
split = int(0.8 * len(indices))
train_idx, val_idx = indices[:split], indices[split:]

train_mean = X[train_idx].mean(axis=0)
train_std = X[train_idx].std(axis=0)
train_std[train_std == 0] = 1.0
X_scaled = (X - train_mean) / train_std

clipped_weight = np.clip(weight, 0.0, np.percentile(weight, 99.5)).astype(np.float32)
clipped_weight = clipped_weight / clipped_weight.mean()

checkpoint = torch.load("outputs/mlp_angles.pt", map_location="cpu", weights_only=False)
model = MLP(X.shape[1], y.shape[1])
model.load_state_dict(checkpoint["model_state"])
model.eval()

with torch.no_grad():
    pred = model(torch.from_numpy(X_scaled[val_idx].astype(np.float32))).numpy()

truth = y[val_idx]
residual = pred - truth

for i, name in enumerate(target_names):
    mae = np.mean(np.abs(residual[:, i]))
    rmse = np.sqrt(np.mean(residual[:, i] ** 2))
    bias = np.mean(residual[:, i])
    corr = np.corrcoef(pred[:, i], truth[:, i])[0, 1]
    print(f"{name}: MAE={mae:.4f}, RMSE={rmse:.4f}, bias={bias:.4f}, corr={corr:.4f}")

## 10. Interpretation of the Current Status

The code currently provides a working baseline. It has successfully connected the ROOT input, event selection, reco/particle-level matching, weighted training, and checkpoint saving.

However, this is still a first-pass regression model. It should be treated as a diagnostic baseline rather than a final physics model.

Important observations:

- `cos_theta_lep` shows a useful learned correlation.
- `cos_theta_star_lep` is more difficult, likely because it depends more strongly on the full event reconstruction and reference-frame definition.
- `phi` and `phi_star` are periodic variables, so direct MSE on the raw angle may not be ideal near the `-pi`/`pi` boundary.
- The final signal/background classifier has not yet been implemented.

## 11. Relationship to Signal/Background Classification

The current four particle-level angular variables are not the final classification output. Instead, they are physics-motivated observables that may carry discriminating information.

Possible next-stage pipeline:

```text
reco-level features
    -> angular regression model
    -> predicted angular observables
    -> signal/background classifier
    -> P(signal)
```

A hybrid classifier could also use both the original reco-level features and the predicted angular variables:

```text
reco-level features + predicted angular variables -> P(signal)
```

This would preserve the original event information while adding interpretable physics features.